# Digit Recognition + SVM + CNN + SVR
This notebook fixes bugs and adds a CNN model.

In [ ]:

# ==============================
# 1. LOAD DATASET (DIGITS)
# ==============================
import numpy as np
import pandas as pd

from sklearn.datasets import load_digits, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.multiclass import OneVsRestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, mean_squared_error, mean_absolute_error

train = pd.read_csv("train.csv");
test = pd.read_csv("test.csv");

X = train.drop(columns=["label"])
y = train["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:

# Linear SVM
linear_svm = SVC(kernel='linear')
linear_svm.fit(X_train, y_train)

pred_linear = linear_svm.predict(X_test)
acc_linear = accuracy_score(y_test, pred_linear)

print("Linear SVM Accuracy:", acc_linear)


Linear SVM Accuracy: 0.9208333333333333


In [ ]:

# RBF SVM
rbf_svm = SVC(kernel='rbf', gamma='scale')
rbf_svm.fit(X_train, y_train)

pred_rbf = rbf_svm.predict(X_test)
acc_rbf = accuracy_score(y_test, pred_rbf)

print("RBF SVM Accuracy:", acc_rbf)


RBF SVM Accuracy: 0.958452380952381


In [ ]:

# PCA + SVM
pca = PCA(n_components=30)

X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

pca_svm = SVC(kernel='rbf')
pca_svm.fit(X_train_pca, y_train)

pred_pca = pca_svm.predict(X_test_pca)
acc_pca = accuracy_score(y_test, pred_pca)

print("PCA + SVM Accuracy:", acc_pca)



PCA + SVM Accuracy: 0.9755952380952381


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, pred_rf)

print("Random Forest Accuracy:", acc_rf)

Random Forest Accuracy: 0.9628571428571429


In [ ]:
# ==============================
# XGBOOST (DIGIT CLASSIFICATION)
# ==============================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

# 1. Load data
train = pd.read_csv("train.csv")

# 2. Separate features and labels
X = train.drop(columns=["label"]).values
y = train["label"].values

# 3. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Create model
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

# 5. Train
xgb.fit(X_train, y_train)

# 6. Predict
y_pred = xgb.predict(X_test)

# 7. Accuracy
acc = accuracy_score(y_test, y_pred)
print("XGBoost Accuracy:", acc)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:04:58] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Accuracy: 0.9738095238095238


In [ ]:
X_small = X_train[:10000]
y_small = y_train[:10000]

# 🔹 Split into train/validation
X_train_small, X_val, y_train_small, y_val = train_test_split(
    X_small, y_small, test_size=0.2, random_state=42
)

# 🔹 Scale
scaler = StandardScaler()
X_train_small = scaler.fit_transform(X_train_small)
X_val = scaler.transform(X_val)

# 🔹 One-vs-Rest SVM
model = OneVsRestClassifier(
    SVC(kernel='linear', C=1)
)

# 🔹 Train
model.fit(X_train_small, y_train_small)

# 🔹 Predict
y_pred = model.predict(X_val)

# 🔹 Accuracy
print("Accuracy:", accuracy_score(y_val, y_pred))

Accuracy: 0.8655


In [ ]:
from sklearn.svm import LinearSVC
import numpy as np

classes = np.unique(y_train)
models = []

for i, c in enumerate(classes):
    print(f"Training class {c} vs rest ({i+1}/{len(classes)})...")

    y_binary = (y_train == c).astype(int)

    clf = LinearSVC(max_iter=5000)
    clf.fit(X_train, y_binary)

    print(f"Finished class {c}")
    models.append(clf)

print("All classifiers trained!")

Training class 0 vs rest (1/10)...
Finished class 0
Training class 1 vs rest (2/10)...
Finished class 1
Training class 2 vs rest (3/10)...


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Finished class 2
Training class 3 vs rest (4/10)...
Finished class 3
Training class 4 vs rest (5/10)...
Finished class 4
Training class 5 vs rest (6/10)...
Finished class 5
Training class 6 vs rest (7/10)...
Finished class 6
Training class 7 vs rest (8/10)...
Finished class 7
Training class 8 vs rest (9/10)...
Finished class 8
Training class 9 vs rest (10/10)...
Finished class 9
All classifiers trained!


In [6]:
scores = []
for clf in models:
    scores.append(clf.decision_function(X_test))

scores = np.array(scores).T  # shape: (samples, classes)

y_pred = np.argmax(scores, axis=1)

# 🔥 ACCURACY
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9039285714285714
